# 09-01A. `matched_next` Label Audit
## Model A를 09-02에 넣기 전 타깃 품질 감사

---

09-01에서 Model A는 성능상 CatBoost가 가장 좋았습니다.

하지만 OOF prediction을 선수 단위로 확인하는 과정에서:

- 다음 시즌 실제 source row가 존재하는데 `matched_next=False`
- 이름 표기 변화 때문에 season-to-season matching이 끊긴 것으로 보이는 사례
- Transfermarkt player ID 자체가 동명이인 때문에 잘못 연결될 수 있는 사례

가 발견되었습니다.

따라서 **09-02 Two-stage Integration 전에 `matched_next`를 감사**합니다.

---

# 이번 Notebook의 목표

### 1. `matched_next` 의미를 다시 고정

프로젝트 데이터 정의:

> 현재 시즌 Big5의 FW/MF 중 900분 이상 출전한 선수를 대상으로  
> 다음 시즌 Big5 기록이 없으면 `next_goals=0`.

`matched_next`는 원래 모델 feature가 아니라
**다음 시즌 레코드 매칭 여부를 확인하기 위한 데이터 품질용 컬럼**입니다.

따라서 Model A target으로 사용하려면
season-to-season entity matching 오류가 없는지 먼저 확인해야 합니다.

### 2. 보수적인 Hard Contradiction만 자동 교정

아래처럼 현재 row가:

```text
matched_next = False
```

인데, target season에 사실상 동일 선수 row가 명확히 존재하면
label contradiction으로 봅니다.

자동 교정은 **보수적으로** 수행합니다.

### 3. 애매한 False는 자동 수정하지 않음

예:

- preseason cutoff 이후 해외 이적
- 장기 부상 / 출장시간 부족
- source 자체 누락 가능성
- transfer date 미확인
- 동명이인

이런 경우는 `audit_status=REVIEW`로만 표시합니다.

---

# 산출물

```text
09_01A_hard_label_corrections.csv
09_01A_matched_next_label_audit.csv
09_01A_audit_summary.csv
09_01A_snapshot_conservative_labels.csv
09_01A_protocol.json
```

원본 `08_v2_preseason_player_snapshot_dev.csv`는 절대 덮어쓰지 않습니다.

# 0. Colab / Google Drive Setup

In [3]:
from pathlib import Path

try:
    from google.colab import drive

    if not Path(
        "/content/drive/MyDrive"
    ).exists():
        drive.mount(
            "/content/drive"
        )
    else:
        print(
            "Google Drive already mounted."
        )

    IN_COLAB = True

except ImportError:
    IN_COLAB = False

print(
    "IN_COLAB:",
    IN_COLAB,
)

Mounted at /content/drive
IN_COLAB: True


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import json
import re
import unicodedata
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option(
    "display.max_columns",
    100,
)

pd.set_option(
    "display.max_rows",
    200,
)

print(
    "pandas:",
    pd.__version__,
)

pandas: 2.2.2


# Part A. Input 탐색

In [7]:
DRIVE_ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/"
    "next_season_goal_prediction/"
    "artifacts"
)

LOCAL_ROOTS = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]


def find_file(
    filename,
    required=True,
):
    drive_candidate = (
        DRIVE_ARTIFACT_DIR
        / filename
    )

    if drive_candidate.exists():
        return drive_candidate

    candidates = []

    for root in (
        LOCAL_ROOTS
    ):
        if not root.exists():
            continue

        try:
            for path in root.rglob(
                filename
            ):
                if path.is_file():
                    candidates.append(
                        path
                    )

        except (
            PermissionError,
            OSError,
        ):
            continue

    if candidates:
        candidates.sort(
            key=lambda p: (
                len(str(p)),
                str(p),
            )
        )

        return candidates[0]

    if required:
        raise FileNotFoundError(
            f"필수 파일을 찾지 못했습니다: {filename}\n"
            f"Drive 예상 위치: {drive_candidate}"
        )

    return None


SNAPSHOT_PATH = find_file(
    "08_v2_preseason_player_snapshot_dev.csv",
    required=True,
)

PREDICTIONS_PATH = find_file(
    "09_01_model_a_predictions.csv",
    required=False,
)

SUMMARY_PATH = find_file(
    "09_01_model_a_summary.csv",
    required=False,
)


if str(
    SNAPSHOT_PATH
).startswith(
    "/content/drive/"
):
    ARTIFACT_DIR = (
        SNAPSHOT_PATH.parent
    )

else:
    ARTIFACT_DIR = Path(
        "artifacts"
    )

    ARTIFACT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


print(
    "Snapshot:",
    SNAPSHOT_PATH,
)

print(
    "09-01 predictions:",
    PREDICTIONS_PATH,
)

print(
    "09-01 summary:",
    SUMMARY_PATH,
)

print(
    "Output:",
    ARTIFACT_DIR.resolve(),
)

Snapshot: /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/08_v2_preseason_player_snapshot_dev.csv
09-01 predictions: /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01_model_a_predictions.csv
09-01 summary: /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01_model_a_summary.csv
Output: /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts


In [8]:
snapshot = pd.read_csv(
    SNAPSHOT_PATH,
    low_memory=False,
)

predictions = (
    pd.read_csv(
        PREDICTIONS_PATH,
        low_memory=False,
    )
    if PREDICTIONS_PATH
    is not None
    else None
)

model_a_summary = (
    pd.read_csv(
        SUMMARY_PATH
    )
    if SUMMARY_PATH
    is not None
    else None
)

print(
    "Snapshot shape:",
    snapshot.shape,
)

display(
    snapshot[
        [
            "row_id",
            "player",
            "team",
            "league",
            "season",
            "target_season",
            "age",
            "minutes",
            "matched_next",
            "next_goals",
            "tm_player_id",
            "match_method",
            "match_confidence",
        ]
    ].head()
)

Snapshot shape: (23353, 114)


,row_id,player,team,league,season,target_season,age,minutes,matched_next,next_goals,tm_player_id,match_method,match_confidence
0,0,Abdelhafid Tasfaout,Guingamp,Ligue 1,2000-2001,2001-2002,31.0,2268.0,True,1.0,155601.0,exact_normalized_unique,A
1,1,Abder Ramdane,Freiburg,Bundesliga,2000-2001,2001-2002,26.0,1491.0,True,0.0,NaN,unmatched,UNMATCHED
2,2,Adaílton,Hellas Verona,Serie A,2000-2001,2001-2002,23.0,1058.0,True,1.0,NaN,unmatched,UNMATCHED
3,3,Ade Akinbiyi,Leicester City,Premier League,2000-2001,2001-2002,25.0,2816.0,True,2.0,4037.0,exact_normalized_unique,A
4,4,Adel Sellimi,Freiburg,Bundesliga,2000-2001,2001-2002,27.0,1851.0,True,5.0,NaN,unmatched,UNMATCHED


# Part B. 원본 데이터 정의 재확인

## 1. 현재 시즌 cohort의 900분 기준 확인

프로젝트 README에서는 현재 시즌 표본을:

```text
Big5
+ FW/MF
+ 900분 이상
```

으로 정의했습니다.

실제 snapshot에서도 season별 최소 minutes를 확인합니다.

> 여기서 중요한 점:
>
> `matched_next=False`가 단순히 "다음 시즌 Big5 밖으로 나감"이라는 뜻은 아닙니다.
>
> next-season raw record 매칭 실패, 이름 문제, 실제 출장 기록 부재 등도 섞일 수 있으므로
> Model A target으로 쓰기 전에 감사가 필요합니다.

In [9]:
minutes_audit = (
    snapshot
    .groupby(
        "season"
    )
    .agg(
        n=(
            "row_id",
            "size",
        ),
        min_minutes=(
            "minutes",
            "min",
        ),
        median_minutes=(
            "minutes",
            "median",
        ),
        max_minutes=(
            "minutes",
            "max",
        ),
    )
    .reset_index()
)

print(
    "Global minimum minutes:",
    snapshot[
        "minutes"
    ].min(),
)

print(
    "Rows below 900 minutes:",
    snapshot[
        "minutes"
    ]
    .lt(900)
    .sum(),
)

display(
    minutes_audit.tail(10)
)

Global minimum minutes: 900.0
Rows below 900 minutes: 0


,season,n,min_minutes,median_minutes,max_minutes
14,2014-2015,1093,900.0,1912.0,3420.0
15,2015-2016,991,900.0,1889.0,3420.0
16,2016-2017,993,901.0,1916.0,3416.0
17,2017-2018,954,900.0,1868.0,3583.0
18,2018-2019,956,901.0,1923.0,3420.0
19,2019-2020,914,901.0,1807.5,3420.0
20,2020-2021,960,905.0,1839.0,3420.0
21,2021-2022,934,904.0,1850.5,3357.0
22,2022-2023,938,903.0,1843.5,3405.0
23,2023-2024,923,900.0,1803.0,3334.0


## 2. Audit 대상 기간

Model A와 동일하게 `target_year >= 2017`만 봅니다.

In [10]:
snapshot[
    "season_start"
] = (
    snapshot[
        "season"
    ]
    .astype(str)
    .str[:4]
    .astype(int)
)

snapshot[
    "target_year"
] = (
    snapshot[
        "target_season"
    ]
    .astype(str)
    .str[:4]
    .astype(int)
)

audit_df = (
    snapshot[
        snapshot[
            "target_year"
        ].ge(2017)
    ]
    .copy()
)

print(
    "Audit rows:",
    len(
        audit_df
    ),
)

print(
    "matched_next=True:",
    audit_df[
        "matched_next"
    ].sum(),
)

print(
    "matched_next=False:",
    (
        ~audit_df[
            "matched_next"
        ].astype(bool)
    ).sum(),
)

Audit rows: 7572
matched_next=True: 6371
matched_next=False: 1201


# Part C. 안전한 문자열 정규화

In [11]:
def normalize_text(
    value,
):
    if pd.isna(
        value
    ):
        return ""

    value = str(
        value
    )

    value = (
        unicodedata
        .normalize(
            "NFKD",
            value,
        )
        .encode(
            "ascii",
            "ignore",
        )
        .decode(
            "ascii"
        )
        .lower()
    )

    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value,
    )

    return " ".join(
        value.split()
    )


snapshot[
    "audit_norm_player"
] = (
    snapshot[
        "player"
    ].map(
        normalize_text
    )
)

snapshot[
    "audit_norm_team"
] = (
    snapshot[
        "team"
    ].map(
        normalize_text
    )
)

audit_df[
    "audit_norm_player"
] = (
    audit_df[
        "player"
    ].map(
        normalize_text
    )
)

audit_df[
    "audit_norm_team"
] = (
    audit_df[
        "team"
    ].map(
        normalize_text
    )
)

# Part D. Hard Contradiction 후보 생성

## 3. 왜 `tm_player_id` 단독으로 자동 수정하면 안 되나?

Transfermarkt crosswalk도 완벽하지 않습니다.

특히:

```text
Adama Traoré
Mariano
Nicolás González
```

처럼 동명이인이 존재하면
같은 normalized name이 잘못된 player ID로 연결될 수 있습니다.

따라서 자동 correction에는 반드시:

- target season
- 나이 연속성
- 이름 유사성
- 팀/ID 보조 정보

를 함께 사용합니다.

### Age rule

다음 시즌 age는 현재 age 대비:

```text
0 ~ +2
```

범위만 허용합니다.

보통은 +1입니다.

In [12]:
AGE_DELTA_MIN = 0
AGE_DELTA_MAX = 2

false_rows = (
    audit_df[
        audit_df[
            "matched_next"
        ].eq(False)
    ]
    .copy()
)

print(
    "False rows to audit:",
    len(
        false_rows
    ),
)

False rows to audit: 1201


## 4. Conservative matching rules

`matched_next=False`인 현재 row와
실제 target-season snapshot row를 비교합니다.

자동 correction candidate는 아래 세 규칙만 허용합니다.

### Rule A — exact_name_age

- normalized player name 동일
- age progression 정상
- 그리고 알려진 Transfermarkt ID가 서로 충돌하지 않음

### Rule B — same_tm_id_name_age

- Transfermarkt ID 동일
- 이름 similarity도 충분히 높음
- age progression 정상

> ID만 같다고 수정하지 않습니다. 동명이인 crosswalk 오류를 막기 위해
> 이름과 나이 sanity check를 함께 통과해야 합니다.

### Rule C — same_team_alias_age

- 같은 team
- age progression 정상
- 이름 spelling/alias 차이만 있는 것으로 보임

예:

```text
Lee Kangin ↔ Lee Kang-in
Martinelli ↔ Gabriel Martinelli
Sepe Elye Wahi ↔ Elye Wahi
```

---

팀이 달라졌는데 이름 spelling도 달라진 케이스는
자동 correction하지 않고 별도 `CROSS_TEAM_ALIAS_REVIEW` 후보로만 저장합니다.

예:

```text
Sehrou Guirassy ↔ Serhou Guirassy
```

**후보가 한 현재 row에 여러 명이면 자동 correction하지 않습니다.**

In [13]:
candidate_records = []

# season별로 미리 분리해서 반복 검색 비용을 줄입니다.
target_by_season = {
    season: group.copy()
    for season, group in (
        snapshot.groupby(
            "season",
            sort=False,
        )
    )
}


def add_candidate(
    store,
    current,
    target,
    rule,
    similarity,
    name_contains,
    same_team,
    same_tm_id,
):
    key = int(
        target[
            "row_id"
        ]
    )

    # 같은 target row가 여러 rule로 잡히면
    # 더 보수적인 우선순위를 유지합니다.
    priority = {
        "exact_name_age": 1,
        "same_tm_id_name_age": 2,
        "same_team_alias_age": 3,
    }

    previous = store.get(
        key
    )

    if (
        previous is not None
        and priority[
            previous[
                "rule"
            ]
        ]
        <= priority[
            rule
        ]
    ):
        return

    store[
        key
    ] = {
        "row_id": (
            current[
                "row_id"
            ]
        ),
        "player": (
            current[
                "player"
            ]
        ),
        "season": (
            current[
                "season"
            ]
        ),
        "target_season": (
            current[
                "target_season"
            ]
        ),
        "team": (
            current[
                "team"
            ]
        ),
        "age": (
            current[
                "age"
            ]
        ),
        "tm_player_id": (
            current[
                "tm_player_id"
            ]
        ),
        "match_confidence": (
            current[
                "match_confidence"
            ]
        ),

        "next_row_id": (
            target[
                "row_id"
            ]
        ),
        "next_player": (
            target[
                "player"
            ]
        ),
        "next_team": (
            target[
                "team"
            ]
        ),
        "next_age": (
            target[
                "age"
            ]
        ),
        "next_tm_player_id": (
            target[
                "tm_player_id"
            ]
        ),
        "next_match_confidence": (
            target[
                "match_confidence"
            ]
        ),
        "recovered_next_goals": (
            target[
                "goals"
            ]
        ),
        "recovered_next_minutes": (
            target[
                "minutes"
            ]
        ),

        "age_delta": (
            target[
                "age"
            ]
            - current[
                "age"
            ]
        ),
        "name_similarity": (
            similarity
        ),
        "name_contains": (
            name_contains
        ),
        "same_team": (
            same_team
        ),
        "same_tm_player_id": (
            same_tm_id
        ),
        "rule": rule,
    }


for _, current in (
    false_rows.iterrows()
):
    target_rows = (
        target_by_season
        .get(
            current[
                "target_season"
            ],
            pd.DataFrame(),
        )
        .copy()
    )

    if target_rows.empty:
        continue

    if pd.notna(
        current[
            "age"
        ]
    ):
        age_delta = (
            target_rows[
                "age"
            ]
            - current[
                "age"
            ]
        )

        target_rows = (
            target_rows[
                age_delta.between(
                    AGE_DELTA_MIN,
                    AGE_DELTA_MAX,
                )
            ]
        )

    if target_rows.empty:
        continue

    current_name = (
        current[
            "audit_norm_player"
        ]
    )

    current_team = (
        current[
            "audit_norm_team"
        ]
    )

    current_id = (
        current[
            "tm_player_id"
        ]
    )

    row_candidates = {}

    # -------------------------------------------------
    # Rule A: exact normalized name + age continuity
    # -------------------------------------------------
    exact_rows = (
        target_rows[
            target_rows[
                "audit_norm_player"
            ].eq(
                current_name
            )
        ]
    )

    for _, target in (
        exact_rows.iterrows()
    ):
        target_id = (
            target[
                "tm_player_id"
            ]
        )

        conflicting_known_ids = (
            pd.notna(
                current_id
            )
            and pd.notna(
                target_id
            )
            and float(
                current_id
            )
            != float(
                target_id
            )
        )

        if conflicting_known_ids:
            continue

        same_tm_id = (
            pd.notna(
                current_id
            )
            and pd.notna(
                target_id
            )
            and float(
                current_id
            )
            == float(
                target_id
            )
        )

        add_candidate(
            row_candidates,
            current,
            target,
            "exact_name_age",
            1.0,
            True,
            (
                current_team
                == target[
                    "audit_norm_team"
                ]
            ),
            same_tm_id,
        )

    # -------------------------------------------------
    # Rule B: same TM ID + name sanity + age continuity
    # -------------------------------------------------
    if pd.notna(
        current_id
    ):
        id_rows = (
            target_rows[
                target_rows[
                    "tm_player_id"
                ].eq(
                    current_id
                )
            ]
        )

        for _, target in (
            id_rows.iterrows()
        ):
            similarity = (
                SequenceMatcher(
                    None,
                    current_name,
                    target[
                        "audit_norm_player"
                    ],
                )
                .ratio()
            )

            if similarity < 0.85:
                continue

            add_candidate(
                row_candidates,
                current,
                target,
                "same_tm_id_name_age",
                similarity,
                (
                    current_name
                    in target[
                        "audit_norm_player"
                    ]
                    or target[
                        "audit_norm_player"
                    ]
                    in current_name
                ),
                (
                    current_team
                    == target[
                        "audit_norm_team"
                    ]
                ),
                True,
            )

    # -------------------------------------------------
    # Rule C: same team + alias/spelling + age continuity
    # -------------------------------------------------
    team_rows = (
        target_rows[
            target_rows[
                "audit_norm_team"
            ].eq(
                current_team
            )
        ]
    )

    for _, target in (
        team_rows.iterrows()
    ):
        target_name = (
            target[
                "audit_norm_player"
            ]
        )

        similarity = (
            SequenceMatcher(
                None,
                current_name,
                target_name,
            )
            .ratio()
        )

        name_contains = (
            (
                current_name
                in target_name
            )
            or (
                target_name
                in current_name
            )
        )

        # 짧은 단일 토큰의 우연한 포함을 완화
        safe_contains = (
            name_contains
            and min(
                len(
                    current_name
                ),
                len(
                    target_name
                ),
            )
            >= 7
        )

        if not (
            similarity
            >= 0.85
            or safe_contains
        ):
            continue

        target_id = (
            target[
                "tm_player_id"
            ]
        )

        same_tm_id = (
            pd.notna(
                current_id
            )
            and pd.notna(
                target_id
            )
            and float(
                current_id
            )
            == float(
                target_id
            )
        )

        add_candidate(
            row_candidates,
            current,
            target,
            "same_team_alias_age",
            similarity,
            name_contains,
            True,
            same_tm_id,
        )

    candidate_records.extend(
        row_candidates.values()
    )


candidates = pd.DataFrame(
    candidate_records
)

print(
    "Candidate matches:",
    len(
        candidates
    ),
)

print(
    "Unique current rows:",
    (
        candidates[
            "row_id"
        ].nunique()
        if not candidates.empty
        else 0
    ),
)

Candidate matches: 20
Unique current rows: 20


## 5. Ambiguous candidate 제거

한 현재 row가 target season의 여러 후보와 연결되면
자동 correction에서 제외합니다.

In [14]:
if candidates.empty:
    candidate_counts = (
        pd.Series(
            dtype=int
        )
    )

    hard_corrections = (
        candidates.copy()
    )

    ambiguous_candidates = (
        candidates.copy()
    )

else:
    candidate_counts = (
        candidates
        .groupby(
            "row_id"
        )
        .size()
    )

    unique_candidate_ids = set(
        candidate_counts[
            candidate_counts.eq(1)
        ].index
    )

    hard_corrections = (
        candidates[
            candidates[
                "row_id"
            ].isin(
                unique_candidate_ids
            )
        ]
        .copy()
    )

    ambiguous_candidates = (
        candidates[
            ~candidates[
                "row_id"
            ].isin(
                unique_candidate_ids
            )
        ]
        .copy()
    )


print(
    "Hard corrections:",
    len(
        hard_corrections
    ),
)

print(
    "Ambiguous candidates:",
    len(
        ambiguous_candidates
    ),
)

display(
    hard_corrections[
        [
            "row_id",
            "player",
            "season",
            "team",
            "age",
            "next_player",
            "next_team",
            "next_age",
            "recovered_next_goals",
            "recovered_next_minutes",
            "name_similarity",
            "same_tm_player_id",
            "rule",
        ]
    ]
)

Hard corrections: 20
Ambiguous candidates: 0


,row_id,player,season,team,age,next_player,next_team,next_age,recovered_next_goals,recovered_next_minutes,name_similarity,same_tm_player_id,rule
0,20600,Amine Adli,2021-2022,Leverkusen,21.0,Amine Adli,Leverkusen,22.0,5.0,1435.0,1.000000,True,exact_name_age
1,20601,Amine Gouiri,2021-2022,Nice,21.0,Amine Gouiri,Rennes,22.0,15.0,2749.0,1.000000,True,exact_name_age
2,20735,Demarai Gray,2021-2022,Everton,25.0,Demarai Gray,Everton,26.0,4.0,2507.0,1.000000,True,exact_name_age
3,20885,Hugo Guillamón,2021-2022,Valencia,21.0,Hugo Guillamón,Valencia,22.0,1.0,1632.0,1.000000,True,exact_name_age
4,20892,Ibrahima Sissoko,2021-2022,Strasbourg,23.0,Ibrahima Sissoko,Strasbourg,24.0,0.0,1310.0,1.000000,True,exact_name_age
5,20975,Jonathan Bamba,2021-2022,Lille,25.0,Jonathan Bamba,Lille,26.0,6.0,2783.0,1.000000,True,exact_name_age
6,21031,Khéphren Thuram-Ulie,2021-2022,Nice,20.0,Khéphren Thuram,Nice,21.0,2.0,2547.0,0.857143,False,same_team_alias_age
7,21053,Lee Kangin,2021-2022,Mallorca,20.0,Lee Kang-in,Mallorca,21.0,6.0,2823.0,0.952381,False,same_team_alias_age
8,21126,Martinelli,2021-2022,Arsenal,20.0,Gabriel Martinelli,Arsenal,21.0,15.0,2789.0,0.714286,False,same_team_alias_age
9,21367,Sepe Elye Wahi,2021-2022,Montpellier,18.0,Elye Wahi,Montpellier,19.0,19.0,2513.0,0.782609,False,same_team_alias_age


## 5-A. Cross-team alias review

팀이 달라졌고 이름 spelling도 바뀐 경우는 자동 correction하지 않습니다.

다만 target-season의 비슷한 이름을 빠르게 찾아
`CROSS_TEAM_ALIAS_REVIEW` 후보로만 저장합니다.

- age progression 정상
- name similarity >= 0.92
- same-team candidate가 아님

이 결과는 사람이 확인한 뒤에만 추가 correction할 수 있습니다.

In [15]:
from difflib import get_close_matches

hard_ids_now = set(
    hard_corrections[
        "row_id"
    ].tolist()
) if not hard_corrections.empty else set()

cross_team_review_records = []

for _, current in (
    false_rows[
        ~false_rows[
            "row_id"
        ].isin(
            hard_ids_now
        )
    ]
    .iterrows()
):
    target_rows = (
        target_by_season
        .get(
            current[
                "target_season"
            ],
            pd.DataFrame(),
        )
        .copy()
    )

    if target_rows.empty:
        continue

    if pd.notna(
        current[
            "age"
        ]
    ):
        age_delta = (
            target_rows[
                "age"
            ]
            - current[
                "age"
            ]
        )

        target_rows = (
            target_rows[
                age_delta.between(
                    AGE_DELTA_MIN,
                    AGE_DELTA_MAX,
                )
            ]
        )

    if target_rows.empty:
        continue

    current_name = (
        current[
            "audit_norm_player"
        ]
    )

    target_names = (
        target_rows[
            "audit_norm_player"
        ]
        .drop_duplicates()
        .tolist()
    )

    close_names = get_close_matches(
        current_name,
        target_names,
        n=2,
        cutoff=0.92,
    )

    for close_name in (
        close_names
    ):
        matches = (
            target_rows[
                target_rows[
                    "audit_norm_player"
                ].eq(
                    close_name
                )
            ]
        )

        for _, target in (
            matches.iterrows()
        ):
            if (
                current[
                    "audit_norm_team"
                ]
                == target[
                    "audit_norm_team"
                ]
            ):
                continue

            similarity = (
                SequenceMatcher(
                    None,
                    current_name,
                    close_name,
                )
                .ratio()
            )

            cross_team_review_records.append({
                "row_id": (
                    current[
                        "row_id"
                    ]
                ),
                "player": (
                    current[
                        "player"
                    ]
                ),
                "season": (
                    current[
                        "season"
                    ]
                ),
                "team": (
                    current[
                        "team"
                    ]
                ),
                "age": (
                    current[
                        "age"
                    ]
                ),
                "next_row_id": (
                    target[
                        "row_id"
                    ]
                ),
                "next_player": (
                    target[
                        "player"
                    ]
                ),
                "next_team": (
                    target[
                        "team"
                    ]
                ),
                "next_age": (
                    target[
                        "age"
                    ]
                ),
                "name_similarity": (
                    similarity
                ),
                "review_status": (
                    "CROSS_TEAM_ALIAS_REVIEW"
                ),
            })


cross_team_alias_review = (
    pd.DataFrame(
        cross_team_review_records
    )
    .sort_values(
        "name_similarity",
        ascending=False,
    )
    if cross_team_review_records
    else pd.DataFrame()
)

print(
    "Cross-team alias review candidates:",
    len(
        cross_team_alias_review
    ),
)

display(
    cross_team_alias_review.head(
        50
    )
)

Cross-team alias review candidates: 1


,row_id,player,season,team,age,next_row_id,next_player,next_team,next_age,name_similarity,review_status
0,21364,Sehrou Guirassy,2021-2022,Rennes,25.0,22323,Serhou Guirassy,Stuttgart,26.0,0.933333,CROSS_TEAM_ALIAS_REVIEW


# Part E. 동명이인 방어 점검

## 6. 위험한 same-name 사례 출력

이 셀은 자동 correction에 들어가지 않은
**age가 크게 어긋나는 same-name target row**를 따로 보여줍니다.

목적:

> `tm_player_id` 또는 exact-name을 맹신하면 안 된다는 것을 확인.

In [16]:
same_name_all = (
    false_rows[
        [
            "row_id",
            "player",
            "season",
            "target_season",
            "team",
            "age",
            "tm_player_id",
            "audit_norm_player",
        ]
    ]
    .merge(
        snapshot[
            [
                "row_id",
                "player",
                "season",
                "team",
                "age",
                "tm_player_id",
                "audit_norm_player",
            ]
        ],
        left_on=[
            "target_season",
            "audit_norm_player",
        ],
        right_on=[
            "season",
            "audit_norm_player",
        ],
        suffixes=[
            "_current",
            "_target",
        ],
    )
)

same_name_all[
    "age_delta"
] = (
    same_name_all[
        "age_target"
    ]
    - same_name_all[
        "age_current"
    ]
)

dangerous_same_name = (
    same_name_all[
        ~same_name_all[
            "age_delta"
        ].between(
            AGE_DELTA_MIN,
            AGE_DELTA_MAX,
        )
    ]
    .copy()
)

display(
    dangerous_same_name[
        [
            "row_id_current",
            "player_current",
            "season_current",
            "team_current",
            "age_current",
            "tm_player_id_current",
            "player_target",
            "team_target",
            "age_target",
            "tm_player_id_target",
            "age_delta",
        ]
    ]
)

,row_id_current,player_current,season_current,team_current,age_current,tm_player_id_current,player_target,team_target,age_target,tm_player_id_target,age_delta
0,16392,Mariano,2016-2017,Sevilla,30.0,54155.0,Mariano,Lyon,23.0,54155.0,-7.0
15,22161,Nicolás González,2022-2023,Valencia,20.0,NaN,Nicolás González,Fiorentina,25.0,NaN,5.0


# Part F. False label 유형 분해

## 7. `matched_next=False`를 네 그룹으로 분리

### HARD_CORRECTION
target-season row가 안전하게 확인됨.

### PRESEASON_CONFIRMED_EXIT
cutoff 이전 transfer가 확인되었고 destination이 Big5 밖.

이건 현재 정보 기준으로 가장 설명 가능한 False입니다.

### LATE_OR_UNKNOWN_TRANSFER_CONTEXT
post-cutoff / 날짜 미확정 transfer context가 존재.

실제 시즌 시작 후 이동했을 가능성이 있어 자동 수정하지 않습니다.

### STABLE_BIG5_REVIEW
cutoff 시점에는 Big5에 남아 있고
명확한 transfer explanation도 없음.

여기에는:

- 실제 출장기록 부족/부상
- raw source 누락
- season matching 실패
- transfer context 누락

가 섞일 수 있으므로 **REVIEW만 하고 자동 correction하지 않습니다.**

In [17]:
hard_ids = set(
    hard_corrections[
        "row_id"
    ].tolist()
)

label_audit = (
    false_rows.copy()
)

label_audit[
    "audit_status"
] = "OTHER_FALSE"

label_audit.loc[
    label_audit[
        "row_id"
    ].isin(
        hard_ids
    ),
    "audit_status",
] = "HARD_CORRECTION"

preseason_exit_mask = (
    label_audit[
        "transfer_event_preseason"
    ].eq(1)
    & label_audit[
        "destination_in_big5"
    ].eq(0)
    & ~label_audit[
        "row_id"
    ].isin(
        hard_ids
    )
)

label_audit.loc[
    preseason_exit_mask,
    "audit_status",
] = "PRESEASON_CONFIRMED_EXIT"

late_or_unknown_mask = (
    (
        label_audit[
            "post_cutoff_context_event_count_audit"
        ]
        .fillna(0)
        .gt(0)
        |
        label_audit[
            "unknown_date_context_event_count_audit"
        ]
        .fillna(0)
        .gt(0)
    )
    & ~label_audit[
        "row_id"
    ].isin(
        hard_ids
    )
    & ~preseason_exit_mask
)

label_audit.loc[
    late_or_unknown_mask,
    "audit_status",
] = "LATE_OR_UNKNOWN_TRANSFER_CONTEXT"

stable_big5_review_mask = (
    label_audit[
        "destination_in_big5"
    ].eq(1)
    & label_audit[
        "transfer_event_preseason"
    ].eq(0)
    & label_audit[
        "post_cutoff_context_event_count_audit"
    ]
    .fillna(0)
    .eq(0)
    & label_audit[
        "unknown_date_context_event_count_audit"
    ]
    .fillna(0)
    .eq(0)
    & ~label_audit[
        "row_id"
    ].isin(
        hard_ids
    )
)

label_audit.loc[
    stable_big5_review_mask,
    "audit_status",
] = "STABLE_BIG5_REVIEW"


status_summary = (
    label_audit[
        "audit_status"
    ]
    .value_counts()
    .rename_axis(
        "audit_status"
    )
    .reset_index(
        name="n"
    )
)

status_summary[
    "rate_among_false"
] = (
    status_summary[
        "n"
    ]
    / len(
        label_audit
    )
)

display(
    status_summary
)

,audit_status,n,rate_among_false
0,STABLE_BIG5_REVIEW,836,0.696087
1,LATE_OR_UNKNOWN_TRANSFER_CONTEXT,200,0.166528
2,PRESEASON_CONFIRMED_EXIT,139,0.115737
3,HARD_CORRECTION,20,0.016653
4,OTHER_FALSE,6,0.004996


## 8. 시즌별 False 구성

In [18]:
false_by_season = (
    label_audit
    .pivot_table(
        index="season",
        columns="audit_status",
        values="row_id",
        aggfunc="count",
        fill_value=0,
    )
)

false_by_season[
    "TOTAL_FALSE"
] = (
    false_by_season.sum(
        axis=1
    )
)

false_by_season

audit_status,HARD_CORRECTION,LATE_OR_UNKNOWN_TRANSFER_CONTEXT,OTHER_FALSE,PRESEASON_CONFIRMED_EXIT,STABLE_BIG5_REVIEW,TOTAL_FALSE
season,,,,,,
2016-2017,0,37,0,17,107,161
2017-2018,0,27,1,9,92,129
2018-2019,0,35,0,10,104,149
2019-2020,0,18,0,5,103,126
2020-2021,0,16,1,17,92,126
2021-2022,11,23,1,18,103,156
2022-2023,9,35,1,31,124,200
2023-2024,0,9,2,32,111,154


# Part G. Stable Big5 Review 우선순위

## 9. 자동 correction하지 않는 이유

`STABLE_BIG5_REVIEW`라고 해서 라벨 오류라고 단정하면 안 됩니다.

예를 들어:

- 시즌 시작 전에는 Big5 팀에 있었지만 이후 이적
- 장기 부상 / 등록 제외
- 실제 next-season 출전이 매우 적음

등은 `matched_next=False`가 맞을 수 있습니다.

따라서 여기서는 **market value / minutes 기준으로 검토 우선순위만 정합니다.**

In [19]:
stable_review = (
    label_audit[
        label_audit[
            "audit_status"
        ].eq(
            "STABLE_BIG5_REVIEW"
        )
    ]
    .copy()
)

stable_review[
    "review_priority_score"
] = (
    stable_review[
        "market_value_preseason_eur"
    ]
    .fillna(0)
    .rank(
        pct=True
    )
    * 0.6
    +
    stable_review[
        "minutes"
    ]
    .fillna(0)
    .rank(
        pct=True
    )
    * 0.4
)

stable_review = (
    stable_review
    .sort_values(
        [
            "review_priority_score",
            "market_value_preseason_eur",
            "minutes",
        ],
        ascending=False,
    )
)

display(
    stable_review[
        [
            "row_id",
            "player",
            "season",
            "target_season",
            "team",
            "age",
            "minutes",
            "goals",
            "market_value_preseason_eur",
            "transfer_event_preseason",
            "destination_team",
            "destination_in_big5",
            "post_cutoff_context_event_count_audit",
            "unknown_date_context_event_count_audit",
            "review_priority_score",
        ]
    ]
    .head(50)
)

,row_id,player,season,target_season,team,age,minutes,goals,market_value_preseason_eur,transfer_event_preseason,destination_team,destination_in_big5,post_cutoff_context_event_count_audit,unknown_date_context_event_count_audit,review_priority_score
22281,22281,Rúben Neves,2022-2023,2023-2024,Wolves,25.0,3019.0,6.0,40000000.0,0,Wolves,1,0,0,0.990909
17754,17754,Aleksandar Mitrović,2018-2019,2019-2020,Fulham,23.0,3280.0,11.0,25000000.0,0,Fulham,1,0,0,0.990191
21316,21316,Rodri,2021-2022,2022-2023,Manchester City,25.0,2884.0,7.0,80000000.0,0,Manchester City,1,0,0,0.987081
16816,16816,Allan,2017-2018,2018-2019,Napoli,26.0,2850.0,4.0,33000000.0,0,Napoli,1,0,0,0.981100
17197,17197,Joe Allen,2017-2018,2018-2019,Stoke City,27.0,3139.0,2.0,18000000.0,0,Stoke City,1,0,0,0.975837
19074,19074,Jefferson Lerma,2019-2020,2020-2021,Bournemouth,24.0,2703.0,1.0,20000000.0,0,Bournemouth,1,0,0,0.962201
18513,18513,Rodrigo,2018-2019,2019-2020,Valencia,27.0,2528.0,8.0,50000000.0,0,Valencia,1,0,0,0.960167
22753,22753,Gustavo Hamer,2023-2024,2024-2025,Sheffield Utd,26.0,2910.0,4.0,15000000.0,0,Sheffield Utd,1,0,0,0.959330
20882,20882,Houssem Aouar,2021-2022,2022-2023,Lyon,23.0,2504.0,6.0,25000000.0,0,Lyon,1,0,0,0.950000
22580,22580,Carlton Morris,2023-2024,2024-2025,Luton Town,27.0,2862.0,11.0,13000000.0,0,Luton Town,1,0,0,0.948804


# Part H. 09-01 OOF와 연결

## 10. CatBoost가 매우 높은 Presence 확률을 준 False

09-01 prediction 파일이 있으면:

```text
실제 matched_next=False
+
CatBoost P(Big5 presence) 높음
```

인 사례가 어떤 audit status인지 확인합니다.

이 분석은:

> 모델이 틀린 것인지,
> label이 의심스러운 것인지

를 분리하는 데 도움이 됩니다.

In [20]:
if predictions is None:
    print(
        "09_01_model_a_predictions.csv가 없어 "
        "OOF error 연결은 건너뜁니다."
    )

else:
    cat_oof = (
        predictions[
            predictions[
                "model"
            ].eq(
                "CatBoost_ModelA"
            )
        ]
        .copy()
    )

    cat_false = (
        cat_oof[
            cat_oof[
                "matched_next"
            ].eq(False)
        ]
        .merge(
            label_audit[
                [
                    "row_id",
                    "audit_status",
                ]
            ],
            on="row_id",
            how="left",
        )
        .sort_values(
            "presence_probability",
            ascending=False,
        )
    )

    display(
        cat_false[
            [
                "player",
                "season",
                "target_season",
                "team",
                "destination_team",
                "destination_in_big5",
                "presence_probability",
                "threshold",
                "audit_status",
            ]
        ]
        .head(50)
    )

,player,season,target_season,team,destination_team,destination_in_big5,presence_probability,threshold,audit_status
131,Amine Gouiri,2021-2022,2022-2023,Nice,Nice,1,0.994929,0.57,HARD_CORRECTION
171,Hugo Guillamón,2021-2022,2022-2023,Valencia,Valencia,1,0.993363,0.57,HARD_CORRECTION
246,Rayan Aït Nouri,2021-2022,2022-2023,Wolves,Wolves,1,0.991248,0.57,STABLE_BIG5_REVIEW
250,Rodri,2021-2022,2022-2023,Manchester City,Manchester City,1,0.990557,0.57,STABLE_BIG5_REVIEW
187,Jens Petter Hauge,2021-2022,2022-2023,Eint Frankfurt,Eintracht Frankfurt,1,0.989261,0.57,LATE_OR_UNKNOWN_TRANSFER_CONTEXT
130,Amine Adli,2021-2022,2022-2023,Leverkusen,Leverkusen,1,0.988811,0.57,HARD_CORRECTION
175,Ibrahima Sissoko,2021-2022,2022-2023,Strasbourg,Strasbourg,1,0.988452,0.57,HARD_CORRECTION
182,Jakub Moder,2021-2022,2022-2023,Brighton,Brighton,1,0.988272,0.57,STABLE_BIG5_REVIEW
291,Alexis Flips,2022-2023,2023-2024,Reims,Reims,1,0.986729,0.71,LATE_OR_UNKNOWN_TRANSFER_CONTEXT
63,Luiz Araújo,2020-2021,2021-2022,Lille,Lille,1,0.985084,0.66,LATE_OR_UNKNOWN_TRANSFER_CONTEXT


# Part I. Conservative label correction

## 11. 원본을 보존한 새 컬럼 생성

중요:

```text
matched_next
next_goals
next_10plus
```

원본은 그대로 둡니다.

새 컬럼:

```text
matched_next_audited
next_goals_audited
next_10plus_audited
label_audit_rule
label_audit_changed
```

만 추가합니다.

Hard contradiction으로 확인된 row만:

```text
False → True
next_goals → target-season row의 실제 goals
```

로 복원합니다.

In [21]:
corrected_snapshot = (
    snapshot.copy()
)

corrected_snapshot[
    "matched_next_original"
] = (
    corrected_snapshot[
        "matched_next"
    ]
)

corrected_snapshot[
    "next_goals_original"
] = (
    corrected_snapshot[
        "next_goals"
    ]
)

corrected_snapshot[
    "next_10plus_original"
] = (
    corrected_snapshot[
        "next_10plus"
    ]
)

corrected_snapshot[
    "matched_next_audited"
] = (
    corrected_snapshot[
        "matched_next"
    ]
    .astype(bool)
)

corrected_snapshot[
    "next_goals_audited"
] = (
    corrected_snapshot[
        "next_goals"
    ]
    .astype(float)
)

corrected_snapshot[
    "next_10plus_audited"
] = (
    corrected_snapshot[
        "next_10plus"
    ]
    .astype(int)
)

corrected_snapshot[
    "label_audit_rule"
] = "UNCHANGED"

corrected_snapshot[
    "label_audit_changed"
] = 0


correction_map = (
    hard_corrections
    .set_index(
        "row_id"
    )
    if not hard_corrections.empty
    else pd.DataFrame()
)


for row_id, correction in (
    correction_map.iterrows()
    if not hard_corrections.empty
    else []
):
    mask = (
        corrected_snapshot[
            "row_id"
        ].eq(
            row_id
        )
    )

    recovered_goals = float(
        correction[
            "recovered_next_goals"
        ]
    )

    corrected_snapshot.loc[
        mask,
        "matched_next_audited",
    ] = True

    corrected_snapshot.loc[
        mask,
        "next_goals_audited",
    ] = recovered_goals

    corrected_snapshot.loc[
        mask,
        "next_10plus_audited",
    ] = int(
        recovered_goals
        >= 10
    )

    corrected_snapshot.loc[
        mask,
        "label_audit_rule",
    ] = correction[
        "rule"
    ]

    corrected_snapshot.loc[
        mask,
        "label_audit_changed",
    ] = 1


print(
    "Rows changed:",
    corrected_snapshot[
        "label_audit_changed"
    ].sum(),
)

Rows changed: 20


## 12. 변경 전/후 class balance

In [22]:
before_after = pd.DataFrame({
    "metric": [
        "development_rows_2017plus",
        "matched_next_true_before",
        "matched_next_true_after",
        "matched_next_false_before",
        "matched_next_false_after",
        "corrections",
    ],

    "value": [
        len(
            audit_df
        ),

        audit_df[
            "matched_next"
        ].sum(),

        corrected_snapshot[
            corrected_snapshot[
                "target_year"
            ].ge(2017)
        ][
            "matched_next_audited"
        ].sum(),

        (
            ~audit_df[
                "matched_next"
            ].astype(bool)
        ).sum(),

        (
            ~corrected_snapshot[
                corrected_snapshot[
                    "target_year"
                ].ge(2017)
            ][
                "matched_next_audited"
            ]
            .astype(bool)
        ).sum(),

        corrected_snapshot[
            corrected_snapshot[
                "target_year"
            ].ge(2017)
        ][
            "label_audit_changed"
        ].sum(),
    ],
})

before_after

,metric,value
0,development_rows_2017plus,7572
1,matched_next_true_before,6371
2,matched_next_true_after,6391
3,matched_next_false_before,1201
4,matched_next_false_after,1181
5,corrections,20


## 13. 복구된 next_goals 확인

In [23]:
recovered_view = (
    corrected_snapshot[
        corrected_snapshot[
            "label_audit_changed"
        ].eq(1)
    ][
        [
            "row_id",
            "player",
            "season",
            "target_season",
            "team",
            "matched_next_original",
            "matched_next_audited",
            "next_goals_original",
            "next_goals_audited",
            "next_10plus_original",
            "next_10plus_audited",
            "label_audit_rule",
        ]
    ]
    .copy()
)

recovered_view

,row_id,player,season,target_season,team,matched_next_original,matched_next_audited,next_goals_original,next_goals_audited,next_10plus_original,next_10plus_audited,label_audit_rule
20600,20600,Amine Adli,2021-2022,2022-2023,Leverkusen,False,True,0.0,5.0,0,0,exact_name_age
20601,20601,Amine Gouiri,2021-2022,2022-2023,Nice,False,True,0.0,15.0,0,1,exact_name_age
20735,20735,Demarai Gray,2021-2022,2022-2023,Everton,False,True,0.0,4.0,0,0,exact_name_age
20885,20885,Hugo Guillamón,2021-2022,2022-2023,Valencia,False,True,0.0,1.0,0,0,exact_name_age
20892,20892,Ibrahima Sissoko,2021-2022,2022-2023,Strasbourg,False,True,0.0,0.0,0,0,exact_name_age
20975,20975,Jonathan Bamba,2021-2022,2022-2023,Lille,False,True,0.0,6.0,0,0,exact_name_age
21031,21031,Khéphren Thuram-Ulie,2021-2022,2022-2023,Nice,False,True,0.0,2.0,0,0,same_team_alias_age
21053,21053,Lee Kangin,2021-2022,2022-2023,Mallorca,False,True,0.0,6.0,0,0,same_team_alias_age
21126,21126,Martinelli,2021-2022,2022-2023,Arsenal,False,True,0.0,15.0,0,1,same_team_alias_age
21367,21367,Sepe Elye Wahi,2021-2022,2022-2023,Montpellier,False,True,0.0,19.0,0,1,same_team_alias_age


# Part J. Data sanity checks

## 14. Correction Assertions

In [24]:
changed = (
    corrected_snapshot[
        corrected_snapshot[
            "label_audit_changed"
        ].eq(1)
    ]
)

assert (
    changed[
        "matched_next_original"
    ].eq(False)
    .all()
)

assert (
    changed[
        "matched_next_audited"
    ].eq(True)
    .all()
)

assert (
    changed[
        "next_goals_audited"
    ].ge(0)
    .all()
)

assert (
    changed[
        "next_10plus_audited"
    ]
    == (
        changed[
            "next_goals_audited"
        ]
        >= 10
    ).astype(int)
).all()

assert (
    corrected_snapshot[
        "row_id"
    ].is_unique
)

print(
    "✅ Conservative correction assertions passed."
)

✅ Conservative correction assertions passed.


# Part K. 결과 저장

In [25]:
OUTPUTS = {
    "hard_corrections": (
        ARTIFACT_DIR
        / "09_01A_hard_label_corrections.csv"
    ),

    "label_audit": (
        ARTIFACT_DIR
        / "09_01A_matched_next_label_audit.csv"
    ),

    "audit_summary": (
        ARTIFACT_DIR
        / "09_01A_audit_summary.csv"
    ),

    "corrected_snapshot": (
        ARTIFACT_DIR
        / "09_01A_snapshot_conservative_labels.csv"
    ),

    "stable_review": (
        ARTIFACT_DIR
        / "09_01A_stable_big5_review.csv"
    ),

    "dangerous_same_name": (
        ARTIFACT_DIR
        / "09_01A_dangerous_same_name_cases.csv"
    ),

    "cross_team_alias_review": (
        ARTIFACT_DIR
        / "09_01A_cross_team_alias_review.csv"
    ),
}


hard_corrections.to_csv(
    OUTPUTS[
        "hard_corrections"
    ],
    index=False,
)

label_audit.to_csv(
    OUTPUTS[
        "label_audit"
    ],
    index=False,
)

status_summary.to_csv(
    OUTPUTS[
        "audit_summary"
    ],
    index=False,
)

corrected_snapshot.to_csv(
    OUTPUTS[
        "corrected_snapshot"
    ],
    index=False,
)

stable_review.to_csv(
    OUTPUTS[
        "stable_review"
    ],
    index=False,
)

dangerous_same_name.to_csv(
    OUTPUTS[
        "dangerous_same_name"
    ],
    index=False,
)

cross_team_alias_review.to_csv(
    OUTPUTS[
        "cross_team_alias_review"
    ],
    index=False,
)


print(
    "Saved:"
)

for name, path in (
    OUTPUTS.items()
):
    print(
        f"- {name:<22}",
        path.resolve(),
    )

Saved:
- hard_corrections       /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_hard_label_corrections.csv
- label_audit            /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_matched_next_label_audit.csv
- audit_summary          /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_audit_summary.csv
- corrected_snapshot     /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_snapshot_conservative_labels.csv
- stable_review          /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_stable_big5_review.csv
- dangerous_same_name    /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_dangerous_same_name_cases.csv
- cross_team_alias_review /content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_cross_team_alias_review.csv


## 15. Protocol 저장

In [26]:
PROTOCOL = {
    "stage": (
        "09-01A matched_next label audit"
    ),

    "source_snapshot": (
        SNAPSHOT_PATH.name
    ),

    "audit_min_target_year": (
        2017
    ),

    "original_target": (
        "matched_next"
    ),

    "original_regression_target": (
        "next_goals"
    ),

    "correction_policy": (
        "conservative; only unique target-season "
        "row matches passing age/name sanity rules"
    ),

    "age_delta_allowed": [
        AGE_DELTA_MIN,
        AGE_DELTA_MAX,
    ],

    "auto_correction_rules": [
        "exact_name_age",
        "same_tm_id_name_age",
        "same_team_alias_age",
    ],

    "do_not_auto_correct": [
        "stable Big5 at cutoff without target row evidence",
        "post-cutoff transfer context",
        "unknown-date transfer context",
        "cross-team alias/spelling candidates",
        "multiple target-season candidate matches",
        "same-name cases with implausible age progression",
    ],

    "new_columns": [
        "matched_next_original",
        "next_goals_original",
        "next_10plus_original",
        "matched_next_audited",
        "next_goals_audited",
        "next_10plus_audited",
        "label_audit_rule",
        "label_audit_changed",
    ],

    "original_snapshot_overwritten": (
        False
    ),

    "next_step": (
        "Review audit outputs, then revalidate Model A "
        "and conditional Model B before 09-02 integration."
    ),
}

PROTOCOL_PATH = (
    ARTIFACT_DIR
    / "09_01A_protocol.json"
)

with PROTOCOL_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        PROTOCOL,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    PROTOCOL_PATH.resolve()
)

/content/drive/MyDrive/next_season_goal_prediction (1)/artifacts/09_01A_protocol.json


# Part L. 결과 해석 체크리스트

실행이 끝나면 아래를 확인합니다.

---

## A. Hard Correction

- 총 몇 건인가?
- 어느 시즌에 집중되는가?
- 어떤 rule이 가장 많은가?
- recovered next goals 중 10+ 득점자가 있는가?

---

## B. Crosswalk 문제

`09_01A_dangerous_same_name_cases.csv`에서:

- 동명이인
- 비정상 age jump
- 동일 ID인데 실제 다른 선수로 보이는 사례

가 있는지 확인합니다.

특히 이 결과는:

> Transfermarkt player ID도 source player-name crosswalk 품질에 의존한다

는 점을 보여줍니다.

---

## C. Stable Big5 Review

이 그룹 전체를 자동 correction하지 않습니다.

확인할 것:

- 상위 market value 선수
- 2,000+ minutes 선수
- no-transfer인데 False인 선수
- 실제 부상/후반 이적/데이터 누락 가능성

---

# 다음 결정

### 경우 1
Hard correction 규모가 작고,
나머지 False가 설명 가능한 경우:

```text
→ audited label로 Model A 재검증
→ Model B도 correction row를 복원하여 재검증
→ 09-02
```

### 경우 2
Stable Big5 Review에서 명백한 source label 오류가 대량 발견되면:

```text
→ Model A target 자체를 재구축
→ 09-02 보류
```

---

## 실행 후 보내줄 파일

우선 아래 4개면 충분합니다.

```text
09_01A_hard_label_corrections.csv
09_01A_audit_summary.csv
09_01A_stable_big5_review.csv
09_01A_dangerous_same_name_cases.csv
09_01A_cross_team_alias_review.csv
```